In [2]:
from dataclasses import dataclass
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_groq import ChatGroq

load_dotenv()

model = ChatGroq(model="openai/gpt-oss-120b")


# Tool Context

In [3]:
@dataclass
class UserContext:
    user_id: str
    tier: str  # e.g., "free" or "pro"


# Simulated account records keyed by user_id
ACCOUNTS = {
    "u_101": {"plan": "Pro", "credits": 240},
    "u_202": {"plan": "Free", "credits": 12},
}


@tool
def get_account_summary(runtime: ToolRuntime[UserContext]) -> str:
    """Look up the current user's account plan and remaining credits."""
    user_id = runtime.context.user_id
    tier = runtime.context.tier
    account = ACCOUNTS.get(user_id, {"plan": "Unknown", "credits": 0})
    return (
        f"User {user_id} ({tier} context): plan={account['plan']}, "
        f"credits={account['credits']}"
    )


agent = create_agent(
    model=model,
    tools=[get_account_summary],
    context_schema=UserContext,
    system_prompt="You are a helpful support assistant. Always use get_account_summary for account questions.",
)

# Pass runtime context on invocation
res1 = agent.invoke(
    {"messages": [{"role": "user", "content": "How many credits do I have left?"}]},
    context=UserContext(user_id="u_101", tier="pro"),
)
print(res1["messages"][-1].content)

# Same agent, different user context—no global state needed
res2 = agent.invoke(
    {"messages": [{"role": "user", "content": "How many credits do I have left?"}]},
    context=UserContext(user_id="u_202", tier="free"),
)
print(res2["messages"][-1].content)



You currently have **240 credits** remaining on your Pro plan. Let me know if you need anything else!
You currently have **12 credits** remaining on your Free plan. Let me know if there's anything else I can help you with!


# Tool Store

In [4]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
store.put(("notes",), "u_101", {"timezone": "Asia/Karachi"})


@tool
def get_user_workspace_info(runtime: ToolRuntime[UserContext]) -> str:
    """Return the user's ID, message count in this run, and saved timezone."""
    msg_count = len(runtime.state.get("messages", []))
    note = runtime.store.get(("notes",), runtime.context.user_id) if runtime.store else None
    tz = note.value.get("timezone") if note else "not set"
    return f"user_id={runtime.context.user_id}, messages={msg_count}, timezone={tz}"


workspace_agent = create_agent(
    model=model,
    tools=[get_user_workspace_info],
    store=store,
    context_schema=UserContext,
)

out = workspace_agent.invoke(
    {"messages": [{"role": "user", "content": "Check my workspace info."}]},
    context=UserContext(user_id="u_101", tier="pro"),
)
print(out["messages"][-1].content)



Here’s the information about your workspace:

- **User ID:** `u_101`  
- **Messages in this session:** `2`  
- **Timezone:** `Asia/Karachi`  

Let me know if you’d like anything else—perhaps a summary of recent activity, a reminder setup, or anything else related to your workspace!


# Dynamic Prompt Middleware

In [5]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    ctx: UserContext = request.runtime.context
    style = "concise and technical" if ctx.tier == "pro" else "clear and beginner-friendly"
    return f"You are assisting user {ctx.user_id}. Keep responses {style}."


prompt_agent = create_agent(
    model=model,
    tools=[],
    middleware=[personalized_prompt],
    context_schema=UserContext,
)

reply = prompt_agent.invoke(
    {"messages": [{"role": "user", "content": "Explain async/await in one short paragraph."}]},
    context=UserContext(user_id="u_202", tier="free"),
)
print(reply["messages"][-1].content)



Async/await is a way to write code that does things that take time (like loading data from the internet) without freezing the rest of your program. When you mark a function as **async**, it can pause its work at any point where you put an **await** in front of a promise‑returning operation. The **await** tells JavaScript (or another language with similar syntax) to “wait here” until that promise resolves, then resume the function as if it were a normal, sequential step. This lets you write asynchronous logic that looks and reads like ordinary, synchronous code, making it easier to understand and maintain.
